Code provided by A. Adame, with some minor changes to include A_sn as a free parameter and to fix certain problems.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import c
from scipy.optimize import curve_fit
import emcee
from scipy import stats
import scipy as sp
import camb 
from getdist import plots, MCSamples
from iminuit import Minuit
from pathlib import Path

In [ ]:
import matplotlib 
font = {'size'   : 20, 'family':'STIXGeneral'}
axislabelfontsize='large'
matplotlib.rc('font', **font)
matplotlib.rcParams['legend.fontsize']='small'
plt.rcParams["figure.figsize"] = [8.0,8.0]

In [ ]:
class mcmc_res:
    def __init__(self, sampler, x0=None):
        self.sampler = sampler
        self.flat_samples = sampler.get_chain(discard=500, thin=15, flat=True)
        self.x = []
        self.x_low = []
        self.x_high = []
        self.ndim = self.flat_samples.shape[1]
        for i in range(self.ndim):
            mcmc = np.percentile(self.flat_samples[:, i], [16, 50, 84])
            q = np.diff(mcmc)
            self.x.append(mcmc[1])
            self.x_low.append(q[0])
            self.x_high.append(q[1])
        self.x0 = x0
        self.nonmarg_bestfit = sampler.flatchain[np.argmin(sampler.flatlnprobability)]
        
def run_mcmc(func_chi2, x0=None, bounds=None, nstep=15000, args=()):    
    assert x0 is not None, 'please, provide a first guess of the parameter values'
    assert bounds is not None, 'please, provide the boundaries of the parameter hyperspace'    
    def log_prior(theta, bounds):
        theta_l = list(theta)
        for i, pp in enumerate(theta_l):
            if (pp < bounds[i][0]) | (pp > bounds[i][1]):
                return -np.inf
        return 0    
    def log_likelihood(theta, *args):
        return -1 * func_chi2(theta, *args)    
    def log_probability(theta, *args):
        lp = log_prior(theta, bounds)
        if not np.isfinite(lp):
            return -np.inf
        return lp + log_likelihood(theta, *args)    
    ndim = len(x0)
    nwalkers = 8 * ndim      
    pos = x0 + 3 * np.random.rand(nwalkers, ndim) 
    sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability, args=args)
    sampler.run_mcmc(pos, nstep, progress=True)    
    result = mcmc_res(sampler, x0=x0) 
    return result

def plot_samples(samples,bestfit_values):
    g = plots.getSubplotPlotter(width_inch=8)
    g.settings.tight_layout = True
    g.settings.axes_labelsize = 20
    g.settings.axes_fontsize = 15
    g.settings.axis_marker_lw = 0.9
    g.settings.axis_marker_color = 'red'
    g.triangle_plot(samples,filled=True, markers=bestfit_values)
    return None

In [ ]:
def delta_growth_factor(z,omega_m0):
    a = 1/(z+1) #Scale factor
    w = -1 #EoS for the Dark Energy. We assume w=-1 for standard DE.
    omega_m = (omega_m0*a**(-3))/(omega_m0*a**(-3)+(1-omega_m0))

    delta =  a*sp.special.hyp2f1( (w-1)/(2*w) , -1/(3*w) ,
                               1- 5/(6*w),1 - 1/omega_m )

    return delta

def growth_factor(z,omega_m0=0.27):
    num = delta_growth_factor(z,omega_m0=omega_m0)
    den = delta_growth_factor(0,omega_m0=omega_m0)
    dz = num/den
    return dz


def get_camb_results(zout,omega_m=0.3089,omega_b=0.048,h=0.6774,ns=0.96,As=2.142e-9,
               s8 = 0.8147,k_pivot=0.05,nonlinear=False,accuracy_boost=1.,kmax=100.):    
    ombh2 = omega_b*h**2
    omch2 = (omega_m-omega_b)*h**2
    
    params = camb.set_params(ns=ns,H0=h*100,ombh2=ombh2,omch2=omch2,
                            As=As,pivot_scalar=k_pivot)
    params.set_matter_power(redshifts=[0],kmax=kmax)
    results = camb.get_results(params)
    s8out = results.get_sigma8()
    As_norm = As*(s8/s8out)**2
    params.set_matter_power(redshifts=[zout],kmax=kmax)
    params.set_accuracy(AccuracyBoost=accuracy_boost)
    results = camb.get_results(params)
    return results
zout = 1.032    # Redshift of the snapshot, you may change this
results = get_camb_results(zout)

def get_pklin(k,camb_results=results):
    pklin = results.get_linear_matter_power_spectrum(hubble_units=True)
    return np.interp(k,pklin[0],pklin[2][0])
Plin = np.vectorize(get_pklin)



transfer_data = results.get_matter_transfer_data()
def get_tk(k,transfer_data=transfer_data):
    kh = np.array(transfer_data.transfer_data[0,:,0])
    tk_raw = np.array(transfer_data.transfer_data[1,:,0])
    tk_raw/=tk_raw[0]
    return np.interp(k,kh,tk_raw)
T = np.vectorize(get_tk)

def get_mk(k,z=zout,omega_m=0.3089,h=0.6774,norm=1.275):
    tk = T(k)
    dz = growth_factor(z)
    c_kms = c/1000.
    num = 2* k**2 *tk*dz *c_kms**2
    den = 3*omega_m*norm *100**2
    return num/den
M = np.vectorize(get_mk)


In [ ]:
'''
Definition of the expressions used in the fit. 
Here, we need the power spectra computed with the notebook provided for it in the repo.
'''

def get_pkhalos(k,b1,bphifnl,pmatter):
    bk = np.ones_like(k)*b1 + bphifnl/M(k)
    return pmatter*bk**2
Pkmodel = np.vectorize(get_pkhalos)

def get_pkcross(k,b1,bphifnl,pmatter):
    bk = np.ones_like(k)*b1 + bphifnl/M(k)
    return pmatter*bk
Pkmodel_cross = np.vectorize(get_pkcross)

def get_sig_pk(k,pk,ngal,sn,L=1000.):
    V = L**3
    n = ngal/V
    delta_k = k[1]-k[0]
    var = 4*np.pi**2 /(V*k**2 *delta_k) *(pk + sn/n)**2 
    return np.sqrt(var)

def get_sig_pkcross(k,pkhalos,pkcross,pkmatter,ngal,sn,L=1000.):
    V = L**3
    n = ngal/V
    delta_k = k[1]-k[0]
    var = 2*np.pi**2 /(V*k**2 *delta_k) *((pkhalos +sn/n)*pkmatter+(pkcross)**2)
    return np.sqrt(var)

In [ ]:
def model_mcmc_b1_fnl_priors(free_params,pmatter,k=None,ngal=None,L=None):
    b1 = free_params[0]
    bphifnl = free_params[1]
    sn = free_params[2]     # Extra shot noise
    n = ngal/(L**3)
    model = Pkmodel(k,b1,bphifnl,pmatter) + sn/n
    return model

def model_mcmc_b1_fnl_priors_cross(free_params,pmatter,k=None,ngal=None,L=None):
    b1 = free_params[0]
    bphifnl = free_params[1]
    n = ngal/(L**3)
    model = Pkmodel_cross(k,b1,bphifnl,pmatter)
    return model

def compute_chi2_mcmc_priors(free_params,pmatter,k,pkhalos,err=None, ngal=None,L=None):
    model = model_mcmc_b1_fnl_priors(free_params,pmatter,k,ngal=ngal,L=L)
    chi2 = np.sum((pkhalos-model)**2 /err**2)   
    return chi2

def compute_chi2_mcmc_priors_cross(free_params,pmatter,k,pkcross,err=None, ngal=None,L=None):
    model = model_mcmc_b1_fnl_priors_cross(free_params,pmatter,k,ngal=ngal,L=L)
    chi2 = np.sum((pkcross-model)**2 /err**2)   
    return chi2

In [ ]:
'''
Computation for the auto-correlation power spectra. Here, as stated in the paper, the shotnoise term is included.
'''
kmax = 0.06

b1prior= 2.0
snprior = 0.0
bphifnlprior = 100.0 * 2*1.686*(b1prior-1)

bins_lims = np.genfromtxt('/path/to/binlims.txt', names=True, delimiter=' ')
bin_inf = bins_lims['lim_inf']
bin_sup = bins_lims['lim_sup']

ndim = 3
nsteps = 15000      # Number of steps for the walkers

array_to_save = []

b1_aux = []
bphifnl_aux = []
sn_aux = []
b1_err_aux = []
bphifnl_err_aux = []
sn_err_aux = []

L = 1000.

for i in range(len(bin_inf)):

    ngal = np.loadtxt('/path/to/N.txt')[i]
    data_halos = np.loadtxt(f'/path/to/pk_halos_selec_{i}.txt')
    k = data_halos[:,0]
    mask = k<kmax

    data_dm = np.loadtxt('/path/to/pk_matter.txt')

    ksel = k[mask]
    pkdm = np.interp(ksel,data_dm[:,0],data_dm[:,1])
    pkhalos = data_halos[mask,1]

    '''
    In the sig_pk formula, there is a dependence on A_sn. 
    As it would be complicated to also fit for A_sn here, we make the
    supposition that in each bin the value of the shotnoise is very similar to the
    one of the following bin.
    For the first bin, the shotnoise is assumed to be poissonian to compute sig_pk.
    '''

    if i==0:
        sig_pk=get_sig_pk(ksel,pkhalos,ngal, sn=1,L=1000.)
    else:
        sig_pk=get_sig_pk(ksel,pkhalos,ngal, sn=sn_aux[i-1],L=1000.)
    
    bounds = [(0,10),(-10000,10000),(-500,500)]
    prior = [b1prior,bphifnlprior,snprior]

    args = (pkdm,ksel,pkhalos,sig_pk,ngal,L)
    result = run_mcmc(compute_chi2_mcmc_priors,x0=prior,bounds=bounds,args=args,nstep=nsteps)

    names=['b1','bphifnl','sn']
    labels=[r'b_1',r'b_\phi f_{\rm NL}',r's_n']

    err = (np.array(result.x_high)+np.array(result.x_low))/2
    bestfit = np.array(result.x)

    samples = MCSamples(samples=result.flat_samples,names=names,labels=labels)   
    plot_samples(samples,bestfit)
    plt.savefig(f'/path/to/store/cornerplot_auto_{i}.png',bbox_inches='tight')
    plt.close()

    print(f'Bin {i}:')
    print('b1 = ',bestfit[0],'+/-',err[0])
    print('bphifnl = ',bestfit[1],'+/-',err[1])
    print('sn = ',bestfit[2],'+/-',err[2])
    print("\n")
    print(samples.getInlineLatex('b1',limit=1))
    print(samples.getInlineLatex('bphifnl',limit=1))
    print(samples.getInlineLatex('sn',limit=1))

    b1_aux=bestfit[0]
    bphifnl_aux=bestfit[1]
    sn_aux=bestfit[2]
    b1_err_aux=err[0]
    bphifnl_err_aux=err[1]
    sn_err_aux=err[2]
    array_to_save.append([b1_aux,bphifnl_aux,sn_aux,b1_err_aux,bphifnl_err_aux,sn_err_aux])  
np.savetxt('/path/to/save/auto_fit.txt',array_to_save, header='b1 bphifnl sn b1_err bphifnl_err sn_err')

In [ ]:
'''
Computation for the cross-correlation power spectra. Here, as stated in the paper, the shotnoise term is NOT present, and in 
the var_pk formula, the shotnoise used is the one determined for the auto-correlation.
'''
kmax = 0.08 

b1prior= 0
bphifnlprior = 100.0 * 2*1.686*(b1prior-1)

bins_lims = np.genfromtxt('path/to/binlims.txt', names=True, delimiter=' ')
bin_inf = bins_lims['lim_inf']
bin_sup = bins_lims['lim_sup']

ndim = 2 
nsteps = 8000       # Number of steps for the walkers

array_to_save = []

b1_aux = []
bphifnl_aux = []
b1_err_aux = []
bphifnl_err_aux = []

L = 1000.

data_auto = np.genfromtxt('/path/to/save/auto_fit.txt', names=True, delimiter=" ")      # For the shotnoise
sn_auto = data_auto['sn']

for i in range(len(bin_inf)):

    ngal = np.loadtxt('/path/to/N.txt')[i]
    data_halos = np.loadtxt(f'/path/to/pk_halos_selec_{i}.txt')
    data_cross = np.loadtxt(f'/path/to/pk_cross_{i}.txt')
    k = data_halos[:,0]
    mask = k<kmax

    data_dm = np.loadtxt('/path/to/pk_matter.txt')

    ksel = k[mask]
    pkdm = np.interp(ksel,data_dm[:,0],data_dm[:,1])
    pkhalos = data_halos[mask,1]
    pkcross = data_cross[mask,1]

    sig_pk=get_sig_pkcross(ksel,pkhalos,pkcross,pkdm,ngal,sn_auto[i],L=1000.)
   
    bounds = [(-10,10),(-10000,10000)]
    prior = [b1prior,bphifnlprior]

    args = (pkdm,ksel,pkcross,sig_pk,ngal,L)
    result = run_mcmc(compute_chi2_mcmc_priors_cross,x0=prior,bounds=bounds,args=args,nstep=nsteps)

    names=['b1','bphifnl']
    labels=[r'b_1',r'b_\phi f_{\rm NL}']

    err = (np.array(result.x_high)+np.array(result.x_low))/2
    bestfit = np.array(result.x)

    samples = MCSamples(samples=result.flat_samples,names=names,labels=labels)   
    plot_samples(samples, bestfit)
    plt.savefig(f'//path/to/store/cornerplot_cross_{i}.png',bbox_inches='tight')
    plt.close()

    print(f'Bin {i}:')
    print('b1 = ',bestfit[0],'+/-',err[0])
    print('bphifnl = ',bestfit[1],'+/-',err[1])
    print("\n")
    print(samples.getInlineLatex('b1',limit=1))
    print(samples.getInlineLatex('bphifnl',limit=1))

    b1_aux=bestfit[0]
    bphifnl_aux=bestfit[1]
    b1_err_aux=err[0]
    bphifnl_err_aux=err[1]
    array_to_save.append([b1_aux,bphifnl_aux,b1_err_aux,bphifnl_err_aux])  

np.savetxt('/path/to/save/cross_fit.txt',array_to_save, header='b1 bphifnl b1_err bphifnl_err')